# BigEarthNet-S2 Dataset Exploration
Visualización y análisis del dataset antes de entrenar el modelo.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import rasterio

from src.data.dataset import CLASSES, CLASS_TO_IDX

In [ ]:
# Ajusta esta ruta a donde tienes el dataset extraído
DATASET_ROOT = Path("/media/alejandro/Almacenamiento HDD/datasets/bigearthnet/BigEarthNet-S2")

## 1. Estructura del dataset

In [ ]:
patches = [p for p in DATASET_ROOT.iterdir() if p.is_dir()]
print(f"Total de patches: {len(patches)}")
print(f"\nEjemplo de patch: {patches[0].name}")
print(f"Archivos dentro: {[f.name for f in patches[0].iterdir()]}")

## 2. Distribución de clases

In [ ]:
class_counts = Counter()

for patch_dir in patches:
    label_file = patch_dir / "labels_metadata.json"
    with open(label_file) as f:
        meta = json.load(f)
    for label in meta["labels"]:
        class_counts[label] += 1

# Ordenar por frecuencia
sorted_classes = sorted(class_counts.items(), key=lambda x: -x[1])

fig, ax = plt.subplots(figsize=(12, 6))
labels, counts = zip(*sorted_classes)
ax.barh(labels, counts, color='steelblue')
ax.set_xlabel('Número de patches')
ax.set_title('Distribución de clases en BigEarthNet-S2')
plt.tight_layout()
plt.show()

## 3. Visualización de imágenes de ejemplo

In [ ]:
def load_rgb(patch_dir: Path) -> np.ndarray:
    """Carga bandas B04, B03, B02 como imagen RGB."""
    bands = []
    for band in ["B04", "B03", "B02"]:
        tif_path = patch_dir / f"{patch_dir.name}_{band}.tif"
        with rasterio.open(tif_path) as src:
            bands.append(src.read(1).astype(np.float32))
    image = np.stack(bands, axis=-1)
    image = image / 10000.0
    image = np.clip(image, 0, 1)
    return image


fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for ax, patch_dir in zip(axes.flatten(), patches[:8]):
    image = load_rgb(patch_dir)
    with open(patch_dir / "labels_metadata.json") as f:
        labels = json.load(f)["labels"]
    ax.imshow(image)
    ax.set_title("\n".join(labels), fontsize=7)
    ax.axis('off')

plt.suptitle('Ejemplos de patches BigEarthNet-S2', fontsize=14)
plt.tight_layout()
plt.show()